# Krones Final Training Notebook

Run this after the ConvNeXt Small and MaxViT Tiny teacher notebooks have exported
their OOF/test predictions. It builds the final EfficientNetV2-S student from
COCO-derived labels, ROI crops, bottle-type metadata, robust noisy-label
weights, and teacher soft labels, then exports `final_effnetv2_s.onnx`.


## Final Strategy

- Use official COCO annotations as the source of truth for training labels.
- Train two diverse teacher models: ConvNeXt Small and MaxViT Tiny.
- Blend teacher OOF probabilities into soft supervision.
- Downweight samples where teachers strongly disagree with the COCO label.
- Train one EfficientNetV2-S student with ROI crop plus bottle-type input.
- Export only the EfficientNetV2-S student to ONNX for final evaluation.

This balances the final score components: hidden-test F1, inference efficiency,
and technical insight.


In [ ]:
from pathlib import Path
import json

STRATEGY_PATH = Path("../configs/final_strategy.json")
if not STRATEGY_PATH.exists():
    STRATEGY_PATH = Path("configs/final_strategy.json")

if STRATEGY_PATH.exists():
    strategy = json.loads(STRATEGY_PATH.read_text())
    print(json.dumps(strategy, indent=2))
else:
    print("Final strategy file not found. Include the strategy summary here before sharing.")


In [ ]:
import subprocess
import sys
from pathlib import Path


def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    return start


ROOT = find_root()
print("ROOT:", ROOT)
print("Python:", sys.executable)


In [ ]:
# Step 1: COCO preprocessing.
# This writes outputs/preprocessing/final_train.csv where target is derived from
# train_annotations.json, not blindly copied from train.csv.
subprocess.check_call([
    sys.executable,
    str(ROOT / "scripts" / "preprocess_coco.py"),
])


In [ ]:
# Step 2: merge ConvNeXt Small + MaxViT Tiny OOF predictions into soft labels.
# If teacher OOF files are not present yet, this still creates the table with
# COCO hard labels and teacher_available=False. For the real final model, rerun
# after both teachers have finished P2 export.
subprocess.check_call([
    sys.executable,
    str(ROOT / "scripts" / "make_teacher_soft_labels.py"),
])


In [ ]:
# Step 3: train final EfficientNetV2-S student and export ONNX.
# On Kaggle GPU, start with EPOCHS=8. On the Mac, use MAX_TRAIN_ROWS for a quick
# smoke run first, then remove it for the real run.
EPOCHS = 8
BATCH_SIZE = 16
MAX_TRAIN_ROWS = None  # set to 512 for a fast local smoke test

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "train_final_student.py"),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", "2",
]
if MAX_TRAIN_ROWS is not None:
    cmd += ["--max-train-rows", str(MAX_TRAIN_ROWS)]

print(" ".join(cmd))
subprocess.check_call(cmd)


In [ ]:
# Step 4: final artifacts expected by final_evaluation.ipynb.
expected = [
    ROOT / "outputs" / "final_effnetv2_s" / "final_effnetv2_s.onnx",
    ROOT / "outputs" / "final_effnetv2_s" / "model_metadata.json",
    ROOT / "outputs" / "final_effnetv2_s" / "teacher_soft_train.csv",
    ROOT / "outputs" / "final_effnetv2_s" / "final_student_summary.json",
]
for path in expected:
    print(("OK  " if path.exists() else "MISS"), path)

summary_path = ROOT / "outputs" / "final_effnetv2_s" / "final_student_summary.json"
if summary_path.exists():
    print(summary_path.read_text())
